In [ ]:
# Create Geolife file
import os
import pandas as pd
from src.calculate_speed import haversine
from src.optimized_analysis import get_prediction_for_trip
from src.process_files_geolife import labels_iterrable, get_filename_for_label_row
from src.stats import is_correct_prediction_custom_thresholds

root_path = 'data/Geolife/'
output_file_path = f'{root_path}database_metrics.csv'

rows = []
count = 0
total_observations_count = 0
for directory in sorted([f for f in os.listdir(root_path) if f.isdigit() and len(f) == 3]):
    user_path = f"{root_path}{directory}/"
    trajectory_path = f'{user_path}Processed_Trajectory/'
    labels_path = f'{user_path}labels.txt'

    for index, row in labels_iterrable(labels_path):
        trajectory_file = get_filename_for_label_row(row, trajectory_path)
        file_to_process = f"{trajectory_path}{trajectory_file}"
        trip_id = f"{directory}.{int(pd.to_datetime(row['Start Time']).strftime('%Y%m%d%H%M%S'))}"

        df = pd.read_csv(file_to_process,
                     skiprows=6,
                     names=['lat', 'lng', '0', 'alt', 'days_since_1899', 'date', 'time'])
        if 'timestamp' not in df.columns:
            df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'])
            df = df.sort_values('timestamp')
            # df = df.set_index('timestamp')

        # Sparsity metric
        df_copy = df.copy()
        df_copy = df_copy.set_index('timestamp')
        datapoints_bucketed_by_minute = df_copy.resample('5s').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        df_copy = df.copy()
        df_copy = df_copy.set_index('timestamp')
        datapoints_bucketed_by_minute = df_copy.resample('30s').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        df_copy = df.copy()
        df_copy = df_copy.set_index('timestamp')
        datapoints_bucketed_by_minute = df_copy.resample('1min').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_1m = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        total_distance_km = haversine(
            df['lat'].shift(1), df['lng'].shift(1),
            df['lat'], df['lng']
        ).sum()

        number_of_records = len(df)
        if df.empty:
            total_trip_time_minutes = 0.0
            density_records_per_minute = 0.0
        else:
            total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
            density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0

        prediction = get_prediction_for_trip(df)
        actual_mode = row['Transportation Mode']
        is_prediction_correct = is_correct_prediction_custom_thresholds(prediction, actual_mode)

        count += 1
        total_observations_count += len(df)
        print(f"Appending row for {trip_id} ({count} files processed)")
        rows.append({
            'trip_id': trip_id,
            'number_of_records': number_of_records,
            'total_trip_time_minutes': round(total_trip_time_minutes, 3),
            'total_distance_km': round(total_distance_km, 3),
            'density_records_per_minute': round(density_records_per_minute, 3),
            'sparsity (per 5s)': round(sparsity_5s, 3),
            'sparsity (per 30s)': round(sparsity_30s, 3),
            'sparsity (per 1m)': round(sparsity_1m, 3),
            'prediction': prediction,
            'actual_mode': actual_mode,
            'is_prediction_correct': is_prediction_correct
        })

dataframe = pd.DataFrame(rows)
dataframe = dataframe.set_index('trip_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f'Total Observations: {total_observations_count}')

# 064.20080831161510
# 065.20110824135121

In [ ]:
# Create rMove database metrics file
import os
import pandas as pd
from src.calculate_speed import haversine
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.stats import is_correct_prediction_custom_thresholds

root_path = 'data/rMove/'
output_file_path = f'{root_path}database_metrics.csv'

locations_df = pd.read_csv(f'{root_path}Location_2023.csv')
trips_df = pd.read_csv(f'{root_path}Household_Travel_Survey_Trips_-7221806773183684102.csv', low_memory=False)
trips_df = trips_df.set_index('trip_id')

rows = []
count = 0

for trip_id, df in locations_df.groupby('tripid'):
    if trip_id not in trips_df.index:
        continue

    df = df.rename(columns={'lon': 'lng', 'collect_time': 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')

        # Sparsity metric
    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('5s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('30s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('1min').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_1m = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    total_distance_km = haversine(
        df['lat'].shift(1), df['lng'].shift(1),
        df['lat'], df['lng']
    ).sum()

    # Calculate density
    number_of_records = len(df)
    if not df.empty:
        total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
        density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
    else:
        total_trip_time_minutes = 0.0
        density_records_per_minute = 0.0

    # Generate prediction and compare to actual mode
    prediction = get_prediction_for_trip_rmove(df)
    trip_info = trips_df.loc[trip_id]
    actual_mode = trip_info['mode_1']
    mapped_prediction = map_to_shared_mode_names(prediction)
    mapped_actual = map_to_shared_mode_names(actual_mode)
    is_prediction_correct = is_correct_prediction_custom_thresholds(mapped_prediction, mapped_actual)

    count += 1
    print(f"Appending row for {trip_id} ({count} files processed)")
    rows.append({
        'trip_id': trip_id,
        'number_of_records': number_of_records,
        'total_trip_time_minutes': round(total_trip_time_minutes, 3),
        'total_distance_km': round(total_distance_km, 3),
        'density_records_per_minute': round(density_records_per_minute, 3),
        'sparsity (per 5s)': round(sparsity_5s, 3),
        'sparsity (per 30s)': round(sparsity_30s, 3),
        'sparsity (per 1m)': round(sparsity_1m, 3),
        'prediction': mapped_prediction,
        'actual_mode': mapped_actual,
        'is_prediction_correct': is_prediction_correct
    })

dataframe = pd.DataFrame(rows)
dataframe = dataframe.set_index('trip_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f"Trips processed: {count}")

In [ ]:
# Create Spectus database metrics file
import pandas as pd
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.calculate_speed import calculate_speed_for_dataframe, haversine

root_path = 'data/Spectus/'
output_file_path = f'{root_path}database_metrics.csv'

locations_df = pd.read_csv(f'{root_path}seattle_2000_processed.csv', low_memory=False)
trips_df = pd.read_csv(f'{root_path}seattle_2000_compressed.csv', low_memory=False)

# Remove stop points
locations_df = locations_df[locations_df['traj_id'] != -99]

locations_df['traj_id'] = locations_df['user_ID'].astype(str) + '_' + locations_df['traj_id'].astype(str)
trips_df['traj_id'] = trips_df['user_ID'].astype(str) + '_' + trips_df['traj_id'].astype(str)

trips_df = trips_df.set_index('traj_id')

rows = []
count = 0
for traj_id, df in locations_df.groupby('traj_id'):
    if traj_id not in trips_df.index:
        continue

    df = df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')

    # Sparsity metric
    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('5s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('30s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('1min').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_1m = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    # Calculate density
    number_of_records = len(df)
    if not df.empty:
        total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
        density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
    else:
        total_trip_time_minutes = 0.0
        density_records_per_minute = 0.0

    total_distance_km = haversine(
        df['lat'].shift(1), df['lng'].shift(1),
        df['lat'], df['lng']
    ).sum()
    _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

    # Generate prediction and compare to actual mode
    prediction = get_prediction_for_trip_rmove(df)
    trip_info = trips_df.loc[traj_id]
    # actual_mode = trip_info['mode_1']
    mapped_prediction = map_to_shared_mode_names(prediction)
    # mapped_actual = map_to_shared_mode_names(actual_mode)
    # is_prediction_correct = is_correct_prediction_custom_thresholds(mapped_prediction, mapped_actual)

    count += 1
    print(f"Appending row for {traj_id} ({count} files processed)")
    rows.append({
        'trip_id': traj_id,
        'number_of_records': number_of_records,
        'total_trip_time_minutes': round(total_trip_time_minutes, 3),
        'total_distance_km': round(total_distance_km, 3),
        'average_speed_kmh': round(average_speed_kmh, 3),
        'max_speed_kmh': round(max_speed_kmh, 3),
        'density_records_per_minute': round(density_records_per_minute, 3),
        'sparsity (per 5s)': round(sparsity_5s, 3),
        'sparsity (per 30s)': round(sparsity_30s, 3),
        'sparsity (per 1m)': round(sparsity_1m, 3),
        'prediction': mapped_prediction
    })

dataframe = pd.DataFrame(rows)
dataframe = dataframe.set_index('trip_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f"Trips processed: {count}")

In [ ]:
# Create Spectus database metrics file
import pandas as pd
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.calculate_speed import calculate_speed_for_dataframe, haversine

root_path = 'data/Spectus/Lyra_Processed/'
output_file_path = f'{root_path}database_metrics.csv'

locations_df = pd.read_csv(f'{root_path}split_by_user/user_0162bdca5925e11a37a48c507453734045b5d62cca0d6abc8300993dfbf8b69e.csv', low_memory=False)
trips_df = pd.read_csv(f'{root_path}Seattle_2000_compressed_trips.csv', low_memory=False)

# Remove stop points
locations_df = locations_df[locations_df['traj_id'] != -99]

locations_df['traj_id'] = locations_df['user_ID'].astype(str) + '_' + locations_df['traj_id'].astype(str)
trips_df['traj_id'] = trips_df['user_ID'].astype(str) + '_' + trips_df['traj_id'].astype(str)

trips_df = trips_df.set_index('traj_id')

rows = []
count = 0
for traj_id, df in locations_df.groupby('traj_id'):
    if traj_id not in trips_df.index:
        continue

    df = df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')

    # Sparsity metric
    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('5s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('30s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    df_copy = df.copy()
    df_copy = df_copy.set_index('timestamp')
    datapoints_bucketed_by_minute = df_copy.resample('1min').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_1m = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    # Calculate density
    number_of_records = len(df)
    if not df.empty:
        total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
        density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
    else:
        total_trip_time_minutes = 0.0
        density_records_per_minute = 0.0

    total_distance_km = haversine(
        df['lat'].shift(1), df['lng'].shift(1),
        df['lat'], df['lng']
    ).sum()
    _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

    # Generate prediction and compare to actual mode
    prediction = get_prediction_for_trip_rmove(df)
    trip_info = trips_df.loc[traj_id]
    # actual_mode = trip_info['mode_1']
    mapped_prediction = map_to_shared_mode_names(prediction)
    # mapped_actual = map_to_shared_mode_names(actual_mode)
    # is_prediction_correct = is_correct_prediction_custom_thresholds(mapped_prediction, mapped_actual)

    count += 1
    print(f"Appending row for {traj_id} ({count} files processed)")
    rows.append({
        'trip_id': traj_id,
        'number_of_records': number_of_records,
        'total_trip_time_minutes': round(total_trip_time_minutes, 3),
        'total_distance_km': round(total_distance_km, 3),
        'average_speed_kmh': round(average_speed_kmh, 3),
        'max_speed_kmh': round(max_speed_kmh, 3),
        'density_records_per_minute': round(density_records_per_minute, 3),
        'sparsity (per 5s)': round(sparsity_5s, 3),
        'sparsity (per 30s)': round(sparsity_30s, 3),
        'sparsity (per 1m)': round(sparsity_1m, 3),
        'prediction': mapped_prediction
    })

dataframe = pd.DataFrame(rows)
dataframe = dataframe.set_index('trip_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f"Trips processed: {count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
import pandas as pd
from pathlib import Path
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.calculate_speed import calculate_speed_for_dataframe, haversine

root_path = 'data/Spectus/Lyra_Processed/split_by_user/'
output_file_path = f'{root_path}database_metrics.csv'

user_count = 0
total_trips_count = 0
for file in Path(root_path).iterdir():
    if not file.is_file():
        continue

    # if user_count >= 10:
    #     break
    count = 0
    try:
        print(f'Processing user {user_count}')
        locations_df = pd.read_csv(f'{root_path}{file.name}', low_memory=False)

        # Remove stop points
        locations_df = locations_df[locations_df['traj_id'] != -99]

        locations_df['traj_id'] = locations_df['user_ID'].astype(str) + '_' + locations_df['traj_id'].astype(str)

        rows = []
        for traj_id, df in locations_df.groupby('traj_id'):

            df = df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df = df.sort_values('timestamp')

            # Sparsity metric
            df_copy = df.copy()
            df_copy = df_copy.set_index('timestamp')
            datapoints_bucketed_by_minute = df_copy.resample('5s').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            df_copy = df.copy()
            df_copy = df_copy.set_index('timestamp')
            datapoints_bucketed_by_minute = df_copy.resample('30s').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            df_copy = df.copy()
            df_copy = df_copy.set_index('timestamp')
            datapoints_bucketed_by_minute = df_copy.resample('1min').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_1m = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            # Calculate density
            number_of_records = len(df)
            if not df.empty:
                total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
                density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
            else:
                total_trip_time_minutes = 0.0
                density_records_per_minute = 0.0

            total_distance_km = haversine(
                df['lat'].shift(1), df['lng'].shift(1),
                df['lat'], df['lng']
            ).sum()
            _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

            # Generate prediction and compare to actual mode
            # prediction = get_prediction_for_trip_rmove(df)
            # mapped_prediction = map_to_shared_mode_names(prediction)

            count += 1

            rows.append({
                'trip_id': traj_id,
                'number_of_records': number_of_records,
                'total_trip_time_minutes': round(total_trip_time_minutes, 3),
                'total_distance_km': round(total_distance_km, 3),
                'average_speed_kmh': round(average_speed_kmh, 3),
                'max_speed_kmh': round(max_speed_kmh, 3),
                'density_records_per_minute': round(density_records_per_minute, 3),
                'sparsity (per 5s)': round(sparsity_5s, 3),
                'sparsity (per 30s)': round(sparsity_30s, 3),
                'sparsity (per 1m)': round(sparsity_1m, 3)
                # 'prediction': mapped_prediction
            })

        dataframe = pd.DataFrame(rows)
        dataframe = dataframe.set_index('trip_id')
        if user_count == 0:
            dataframe.to_csv(output_file_path, index=True, header=True, mode='w')
        else:
            dataframe.to_csv(output_file_path, index=True, header=False, mode='a')
    except Exception as e:
        print(f"Unexpected error: {e}")
        print(f"file: {file.name}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Trips processed: {count}")